In [1]:
!pip install scipy

You should consider upgrading via the 'C:\Users\User\Documents\TKU\大二AI實驗下\DL_ai\Scripts\python.exe -m pip install --upgrade pip' command.


## Part1. Inception Block

FLOPs = kernel_H × kernel_W × in_channels × out_channels × H × W

In [ ]:
import torch
import torch.nn as nn
from thop import profile #計算量估計

1. Normal conv

In [ ]:
class NormalBlock(nn.Module):
    def __init__(self, in_channel, out_channel):
        super().__init__()
        self.conv = nn.Conv2d(in_channel, out_channel, kernel_size=5, padding=2)
    
    def forward(self, x):
        return self.conv(x)

2.Inception Block

In [ ]:
class InceptionOriginal(nn.Module):
    def __init__(self, inchannel):
        super().__init__()
        self.b1 = nn.Conv2d(inchannel, 32, kernel_size=1)
        self.b2 = nn.Conv2d(inchannel, 32, kernel_size=3, padding=1)
        self.b3 = nn.Conv2d(inchannel, 32, kernel_size=5, padding=2)
        self.b4 = nn.MaxPool2d(3, stride=1, padding=1)
        self.b4_conv = nn.Conv2d(inchannel, 32, kernel_size=1)

    #串聯網路 (Parallel)
    def forward(self, x):
        return torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4_conv(self.b4(x))], 1)

3.Paper改良

In [ ]:
class GoogleInception(nn.Module):
    def __init__(self, in_channel):
        super().__init__()
        self.b1 = nn.Conv2d(in_channel, 32, kernel_size=1)
        self.b2 = nn.Sequential(
            nn.Conv2d(in_channel, 16,kernel_size=1),
            nn.Conv2d(16, 32, kernel_size=3, padding=1)
        )
        self.b3 = nn.Sequential(
            nn.Conv2d(in_channel, 16, kernel_size=1),
            nn.Conv2d(16, 32, kernel_size=5, padding=2)
        )
        self.b4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_channel, 32, kernel_size=1)
        )

    def forward(self, x):
        return torch.cat([self.b1(x), self.b2(x), self.b3(x),self.b4(x)], 1)

實驗:

In [ ]:
input_data = torch.randn(1, 64, 28, 28)

models = [
    ("NormalBlock", NormalBlock(64, 128)),
    ("InceptionOriginal", InceptionOriginal(64)),
    ("GoogleInception", GoogleInception(64))
]
print(f"{'架構名稱':<30} | {'參數 (Params)':<15} | {'運算量 (FLOPs)':<15}")
for name, model in models:
    flops, params = profile(model, inputs =(input_data, ),verbose=False)
    print(f"{name:<33} | {params:<17.0f} | {flops/1e6:} M")

## Part2. GoogLeNet

In [ ]:
import torch 
import torch.nn as nn
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from tqdm import tqdm

import numpy as np


In [ ]:
transform_train = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),          # 資料增強
    transforms.RandomHorizontalFlip(),   # 資料增強
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
from torch.utils.data import Subset
train_dataset = torchvision.datasets.Food101(
    root='./data',
    split='train',
    download=True,
    transform=transform_train
)

test_dataset = torchvision.datasets.Food101(
    root='./data',
    split='test',
    download=True,
    transform=transform_test
)
train_indices = np.random.choice(len(train_dataset), 1000, replace=False)
test_indices  = np.random.choice(len(test_dataset),  500, replace=False)
train_subset = Subset(train_dataset, train_indices)
test_subset  = Subset(test_dataset,  test_indices)
train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_subset,  batch_size=32, shuffle=False)

影像排列: (Batch, Channel, Height, Width)
            0        1       2        3

### Block工具製作

In [ ]:
#======================= conv block =======================





#======================= Inception block =======================






#======================= InceptionAux =======================






### 定義本網路架構

In [ ]:
#======================= GoogLeNet =======================

### 設定網路參數

In [ ]:
model = GoogLeNet(aux_logits=True, num_classes=101)
model = model.to('cuda')

optimizer = optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

### 開始Train

In [ ]:
from tqdm import tqdm

for epoch in range(100):
    model.train()
    
    total_loss  = 0
    num_correct = 0
    num_samples = 0

    loop = tqdm(train_loader, desc=f'Epoch [{epoch+1}/100]')

    for batch_idx, (data, targets) in enumerate(loop):
        data    = data.to("cuda")
        targets = targets.to("cuda")

        scores = model(data)

        loss0 = criterion(scores[0], targets)
        loss1 = criterion(scores[1], targets)
        loss2 = criterion(scores[2], targets)
        loss  = loss0 * 0.3 + loss1 * 0.3 + loss2
        #當aux = false時
        #loss = criterion(scores, targets) 
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss  += loss.item()
        _, predicted = scores[2].max(1)
        #當aux = false時
        #_, predicted = scores.max(1) 
        num_correct += (predicted == targets).sum().item()
        num_samples += targets.size(0)

        loop.set_postfix(
            loss = f'{loss.item():.4f}',
            acc  = f'{100 * num_correct / num_samples:.2f}%'
        )

    avg_loss = total_loss / len(train_loader)
    avg_acc  = 100 * num_correct / num_samples
    print(f'Epoch [{epoch+1}/100] Loss: {avg_loss:.4f} | Train Acc: {avg_acc:.2f}%')

    # ── 驗證 ──────────────────────────────────
    model.eval()
    val_correct = 0
    val_samples = 0
    val_loss    = 0                                         

    with torch.no_grad():
        for data, targets in test_loader:
            data    = data.to("cuda")
            targets = targets.to("cuda")
            scores  = model(data)

            loss = criterion(scores, targets)              
            val_loss += loss.item()                         

            _, predicted = scores.max(1)
            val_correct += (predicted == targets).sum().item()
            val_samples += targets.size(0)

    avg_val_loss = val_loss / len(test_loader)              
    val_acc      = 100 * val_correct / val_samples
    print(f'           Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%')  
    print('-' * 45)

Epoch [1/100]: 100%|██████████| 157/157 [00:50<00:00,  3.09it/s, acc=3.24%, loss=7.5807]


Epoch [1/100] Loss: 7.3021 | Train Acc: 3.24%
           Val Loss: 4.4128 | Val Acc: 4.00%
---------------------------------------------


Epoch [2/100]: 100%|██████████| 157/157 [00:30<00:00,  5.23it/s, acc=5.34%, loss=6.3541]


Epoch [2/100] Loss: 6.9327 | Train Acc: 5.34%
           Val Loss: 4.3432 | Val Acc: 5.30%
---------------------------------------------


Epoch [3/100]: 100%|██████████| 157/157 [00:29<00:00,  5.24it/s, acc=7.30%, loss=6.2138]


...

Epoch [98/100] Loss: 0.3600 | Train Acc: 96.48%
           Val Loss: 6.1324 | Val Acc: 24.00%
---------------------------------------------


Epoch [99/100]: 100%|██████████| 157/157 [00:28<00:00,  5.46it/s, acc=97.06%, loss=5.1441]


Epoch [99/100] Loss: 0.3228 | Train Acc: 97.06%
           Val Loss: 6.4869 | Val Acc: 22.30%
---------------------------------------------


Epoch [100/100]: 100%|██████████| 157/157 [00:28<00:00,  5.43it/s, acc=93.46%, loss=0.8535]


Epoch [100/100] Loss: 0.5187 | Train Acc: 93.46%
           Val Loss: 6.4192 | Val Acc: 21.90%
---------------------------------------------


## 3.本日作業

繳交格式: 問題請直接使用文字回答即可，實作相關請附程式碼截圖證明還有文字說明

#### P1.Inception Block問題

1. 請問將input的channel，或是output的channel變成兩倍，對於我們的計算FLOPs會有甚麼影響?
2. 今天我in_channel=192, out_channel=128, 打算用3x3的捲積核。請問(1)直接使用3x3捲積與(2)先用1x1降到96後再3x3捲積輸出到128，兩者差多少參數? (參考公式，大概數值即可)

#### P2.GoogLeNet實作挑戰

1. 請設法調整訓練參數，讓訓練的過程中train acc與test acc均高於 45%<br>
Epoch [9/100] Loss: 3.2517 | Train Acc: 52.17%<br>
           Val Loss: 1.8459 | Val Acc: 51.70%<br>
---------------------------------------------<br>

2. 跑aux_logits=True, aux_logits=False,比較學習的趨勢與準確率<br>
注意:aux=false，training的程式會需要改兩個地方，請看註解處